# Libraries and global variables

In [ ]:
import sys
import time
from pathlib import Path
import json

import numpy as np

import torch
from fastai.vision.all import (
    DataBlock, ImageBlock,
    ColReader, ColSplitter,
    vision_learner,
    SaveModelCallback, EarlyStoppingCallback, CSVLogger,
    accuracy, RocAuc, F1Score, ClassificationInterpretation,
    Resize, DataLoaders
)
from fastai.callback.fp16 import MixedPrecision
import timm  # needed for EfficientNet via fastai

from fastai.vision.augment import (
    Brightness, Contrast, aug_transforms
)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!cp -r "/content/drive/MyDrive/Work/10 Foodbegood/project_gn_food_estimator_v1" "/content/project_gn_food_estimator_v1"

## Directory layout & paths

In [ ]:
PLATFORM = "colab" # "colab", "kaggle", "local"

if PLATFORM == "colab":
    ROOT = Path("/content/project_gn_food_estimator_v1")
elif PLATFORM == "kaggle":
    ROOT = Path("/kaggle/working/project_gn_food_estimator")
else:
    ROOT = Path(".").resolve()

In [ ]:
DATA_DIR      = ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORTS_DIR   = DATA_DIR / "exports"
MODELS_DIR    = ROOT / "models"
TRAINING_DIR  = ROOT / "training"
LOGS_DIR      = TRAINING_DIR / "logs"

# Labels file
COCO_MODEL2_JSON = EXPORTS_DIR / "model2_fill_level.json"

# Model checkpoint path
MODEL2_PATH = MODELS_DIR / "model2_fill.pth"

In [ ]:
def setup_directories():
    """
    Ensure output directories exist.
    """
    MODELS_DIR.mkdir(parents = True, exist_ok = True)
    LOGS_DIR.mkdir(parents = True, exist_ok = True)
    print(f"[Setup] Models dir: {MODELS_DIR}")
    print(f"[Setup] Logs dir : {LOGS_DIR}")

In [ ]:
setup_directories()

## Parameters

In [ ]:
# Dataset split ratios
SPLIT_TRAIN = 0.80
SPLIT_VAL   = 0.10
SPLIT_TEST  = 0.10
RANDOM_SEED = 42

In [ ]:
# Class labels — must match Label Studio annotation labels exactly
MODEL2_CLASSES = ["empty", "low", "medium", "high", "full"]

In [ ]:
# If classifier confidence falls below this threshold, ask user to retake photo
CONFIDENCE_THRESHOLD = 0.70

In [ ]:
# Image pre-processing
IMAGE_SIZE = 512
NORM_MEAN = [0.485, 0.456, 0.406] # ImageNet statistics
NORM_STD = [0.229, 0.224, 0.225]
HIST_EQ_ENABLED = True

In [ ]:
# Data augmentation toggles
AUG_BRIGHTNESS = True
AUG_ROTATION   = True
AUG_BLUR       = True
AUG_SCALING    = True
AUG_FLIPPING   = True
AUG_CONTRAST   = True

## Training hyperparameters

In [ ]:
# Model #2 — EfficientNet-B0, image classification
MODEL2_BACKBONE        = "efficientnet_b0"
MODEL2_BATCH_SIZE      = 16      # Lighter model, larger batch fine on T4
MODEL2_EPOCHS_FROZEN   = 5
MODEL2_EPOCHS_UNFROZEN = 10
MODEL2_LR_FROZEN       = 1e-3
MODEL2_LR_UNFROZEN     = 1e-4
MODEL2_MIXED_PREC      = True

In [ ]:
def check_gpu():
    """
    Report GPU availability and VRAM.
    """
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

        print(f"[GPU] {gpu_name} - {vram_gb:.1f} GB VRAM")

        if vram_gb < 8.0:
            print(
                f"[GPU] WARNING: Less than 8 GB VRAM detected. "
                f"Consider reducing MODEL2_BATCH_SIZE."
                f" (currently {MODEL2_BATCH_SIZE})."
            )
    else:
        print("[GPU] No CUDA device found - training will run on CPU.")
    return "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
DEVICE = check_gpu()

# Data Preprocessing

## Data augmentation pipeline

In [ ]:
def get_classification_transforms(size: int = IMAGE_SIZE):
    """
    fastai's aug_transforms() covers: flip, rotation, zoom (scaling),
    warp, lighting (brightness + contrast), and blur.

    Returns
        tuple[list, list]
            (train_transforms, valid_transforms)
    """
    aug_kwargs = dict(
        size = size,

        # Flipping: only horizontal — vertical flip is disabled for top-down food
        do_flip = AUG_FLIPPING,
        flip_vert = False,

        # Rotation: ±20° captures realistic hand-angle variation
        max_rotate = 20.0 if AUG_ROTATION else 0.0,

        # Scaling / zoom: 1.0–1.15 — mild zoom to simulate distance variation
        min_zoom = 1.0,
        max_zoom = 1.15 if AUG_SCALING else 1.0,

        # Lighting (brightness + contrast handled together by aug_transforms)
        max_lighting = 0.3 if (AUG_BRIGHTNESS or AUG_CONTRAST) else 0.0,

        # Blur: p = 0.3 means applied to ~30% of batches
        max_blur = 2.0 if AUG_BLUR else 0.0,

        # Warp: small perspective warp simulates slight camera tilt
        max_warp = 0.1,

        p_affine = 0.75,   # probability of applying affine transforms
        p_lighting = 0.75, # probability of applying lighting transforms
    )

    train_tfms = aug_transforms(**aug_kwargs)
    valid_tfms = []  # No augmentation on validation / test sets

    return train_tfms, valid_tfms

In [ ]:
train_tfms, _ = get_classification_transforms(size = IMAGE_SIZE)

# Loading data

## Build DataLoaders

In [ ]:
# Building DataLoaders
dls, train_df, test_df = get_classification_dataloaders(
        coco_json_path = COCO_MODEL2_JSON,
        image_dir = PROCESSED_DIR,
        image_size = IMAGE_SIZE,
        batch_size = MODEL2_BATCH_SIZE,
        train_tfms = train_tfms,
        device = DEVICE,
        verbose = True,
    )

In [ ]:
# Verify vocab matches config
if list(dls.vocab) != MODEL2_CLASSES:
    print(
        f"\n  WARNING: DataLoaders vocab {list(dls.vocab)} does not match "
        f"MODEL2_CLASSES {MODEL2_CLASSES}.\n"
        f"  Predictions will use DataLoaders vocab ordering."
    )

In [ ]:
# Save test split
test_csv_path = LOGS_DIR / "model2_test_split.csv"
test_df.to_csv(test_csv_path, index = False)
print(f"Test split saved → {test_csv_path}")

In [ ]:
# Sample batch sanity check
xb, yb = dls.one_batch()
print(f"Sample batch — images: {xb.shape}, labels: {yb.shape}")

In [ ]:
# Class weight helper
def compute_class_weights(train_df, label_col: str = "label") -> torch.Tensor:
    """
    Compute inverse-frequency class weights to handle any class imbalance.

    With only 100 photos split 80/10/10 across 5 classes × 2 container
    types × 2 food types, some fill levels may be slightly underrepresented.
    Passing weights to CrossEntropyLossFlat helps.

    Returns
        torch.Tensor of shape (n_classes,)
    """
    counts = train_df[train_df["split"] == "train"][label_col].value_counts()

    # Ensure ordering matches MODEL2_CLASSES
    ordered_counts = [counts.get(cls, 1) for cls in MODEL2_CLASSES]
    total = sum(ordered_counts)
    weights = torch.tensor(
        [total / (len(MODEL2_CLASSES) * c) for c in ordered_counts],
        dtype = torch.float32,
    )

    print("Class weights:")
    for cls, w in zip(MODEL2_CLASSES, weights):
        print(f"    {cls:8s} : {w:.3f}")

    return weights

In [ ]:
# Computing class weights (handle minor imbalance)
class_weights = compute_class_weights(train_df)
if device == "cuda":
    class_weights = class_weights.cuda()

# Model

## Freeze / unfreeze helpers

# Training loop

## Mixed precision scaler

## Phase 1 - frozen backbone

## Phase 2 - full fine-tune

## Save training log

# Export model

# Quick inference check

## Visualisation